# Imports

In [ ]:
import json
from collections import deque
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx            # preverjanje in izvoz v GraphML

rng = np.random.default_rng(1)
IZVOZ = Path("izvoz")            # mapa za izvožene grafe
IZVOZ.mkdir(exist_ok=True)

# Funkcije

In [ ]:
# --- Graf in metrike z vaje 07 ---
class Graf:
    def __init__(self):
        self.sosedje = {}

    def dodaj_vozlisce(self, v):
        self.sosedje.setdefault(v, set())

    def dodaj_povezavo(self, u, v):
        if u == v:
            return
        self.dodaj_vozlisce(u)
        self.dodaj_vozlisce(v)
        self.sosedje[u].add(v)
        self.sosedje[v].add(u)

    def odstrani_povezavo(self, u, v):
        self.sosedje[u].discard(v)
        self.sosedje[v].discard(u)

    def n(self):
        return len(self.sosedje)

    def m(self):
        return sum(len(s) for s in self.sosedje.values()) // 2

    def stopnja(self, v):
        return len(self.sosedje[v])

    def povezave(self):
        return [(u, v) for u in self.sosedje for v in self.sosedje[u] if u < v]

    def __repr__(self):
        return f"Graf(n={self.n()}, m={self.m()})"


def lokalna_grucavost(G, v):
    s = list(G.sosedje[v])
    k = len(s)
    if k < 2:
        return 0.0
    e = sum(1 for i in range(k) for j in range(i + 1, k) if s[j] in G.sosedje[s[i]])
    return 2 * e / (k * (k - 1))


def povprecna_grucavost(G):
    return np.mean([lokalna_grucavost(G, v) for v in G.sosedje])


def bfs(G, izvor):
    d = {izvor: 0}
    vrsta = deque([izvor])
    while vrsta:
        u = vrsta.popleft()
        for w in G.sosedje[u]:
            if w not in d:
                d[w] = d[u] + 1
                vrsta.append(w)
    return d


def povprecna_pot(G):
    """Povprečna najkrajša pot na komponenti, ki vsebuje vozlišče 0."""
    vsota, parov = 0, 0
    for v in bfs(G, 0):
        d = bfs(G, v)
        vsota += sum(d.values())
        parov += len(d) - 1
    return vsota / parov


def obroc(n, k):
    G = Graf()
    for i in range(n):
        for j in range(1, k // 2 + 1):
            G.dodaj_povezavo(i, (i + j) % n)
    return G


# --- Generatorji ---
def watts_strogatz(n, k, p, rng=rng):
    """Obroč, v katerem vsako povezavo (i, i+j) z verjetnostjo p prevežemo na naključno vozlišče."""
    G = obroc(n, k)
    for j in range(1, k // 2 + 1):
        for i in range(n):
            u, v = i, (i + j) % n
            if rng.random() < p and v in G.sosedje[u]:
                # TODO: izberi novo krajišče w (ne u in ne obstoječi sosed), zamenjaj (u, v) z (u, w)
                pass
    return G


def barabasi_albert(n, m, rng=rng):
    """Preferenčno pripenjanje: vsako novo vozlišče se poveže z m obstoječimi,
    verjetnost izbire je sorazmerna s stopnjo."""
    G = Graf()
    for i in range(m + 1):                 # začetni poln graf z m + 1 vozlišči
        for j in range(i + 1, m + 1):
            G.dodaj_povezavo(i, j)
    krajisca = [v for e in G.povezave() for v in e]   # vsako vozlišče je zapisano k-krat
    for novo in range(m + 1, n):
        # TODO: izberi m različnih ciljev iz seznama krajišč (sorazmerno s stopnjo), dodaj povezave in posodobi seznam
        pass
    return G


# --- Izris ---
def krozna_razporeditev(G):
    v = sorted(G.sosedje)
    fi = 2 * np.pi * np.arange(len(v)) / len(v)
    return {u: np.array([np.cos(f), np.sin(f)]) for u, f in zip(v, fi)}


def fr_razporeditev(G, n_iter=200, rng=rng):
    """Fruchterman-Reingoldova razporeditev: odboj k^2/d med vsemi pari, privlak d^2/k po povezavah."""
    voz = sorted(G.sosedje)
    ind = {v: i for i, v in enumerate(voz)}
    E = np.array([(ind[u], ind[v]) for u, v in G.povezave()])
    N = len(voz)
    poz = rng.uniform(-1, 1, (N, 2))
    k = np.sqrt(4.0 / N)                       # optimalna razdalja (ploščina 2 x 2)
    T = 0.2
    for it in range(n_iter):
        dr = poz[:, None, :] - poz[None, :, :]
        d = np.linalg.norm(dr, axis=-1) + 1e-9
        # TODO: odbojne sile (k^2/d) med vsemi pari in privlačne (d^2/k) po povezavah
        premik = np.zeros_like(poz)
        dolz = np.linalg.norm(premik, axis=1)[:, None] + 1e-9
        poz += premik / dolz * np.minimum(dolz, T)  # premik omejimo s temperaturo
        T *= 0.98                                    # ohlajanje
    return {v: poz[ind[v]] for v in voz}


def narisi_graf(ax, G, poz, velikost=None):
    for u, v in G.povezave():
        ax.plot(*np.array([poz[u], poz[v]]).T, color="gray", lw=0.5, zorder=1)
    st = np.array([G.stopnja(v) for v in poz])
    xy = np.array(list(poz.values()))
    ax.scatter(xy[:, 0], xy[:, 1], s=velikost if velikost is not None else 3 + 2 * st,
               c=st, cmap="viridis", zorder=2)
    ax.set_aspect("equal")
    ax.axis("off")


# --- Izvoz in uvoz ---
def izvozi_csv(G, pot):
    # TODO: vrstica "u,v" za vsako povezavo (z glavo "vir,cilj")
    pass


def uvozi_csv(pot):
    G = Graf()
    # TODO
    pass
    return G


def izvozi_json(G, pot):
    # TODO: slovar {"vozlisca": [...], "povezave": [[u, v], ...]} z json.dump
    pass


def uvozi_json(pot):
    G = Graf()
    # TODO
    pass
    return G


def v_networkx(G):
    H = nx.Graph()
    H.add_nodes_from(G.sosedje)
    H.add_edges_from(G.povezave())
    return H

# Testno območje

In [ ]:
G0 = obroc(500, 10)
print(G0, f"C(0) = {povprecna_grucavost(G0):.3f}, L(0) = {povprecna_pot(G0):.2f}")
Gw = watts_strogatz(500, 10, 0.01)
print("WS p = 0.01:", Gw, "| prevezanih ≈", len(set(G0.povezave()) - set(Gw.povezave())))
Gb = barabasi_albert(2000, 2)
print("BA:", Gb, f"<k> = {2*Gb.m()/Gb.n():.2f}, k_max = {max(Gb.stopnja(v) for v in Gb.sosedje)}")

# Naloga 1 - Slika 1

In [ ]:
p_v = np.logspace(-4, 0, 13)
C0, L0 = povprecna_grucavost(G0), povprecna_pot(G0)
# TODO: C(p)/C(0) in L(p)/L(0) za n = 500, k = 10 (povprečje 2 ponovitev)
C_p = L_p = None
fig = plt.figure(figsize=(7, 4))
ax = fig.add_subplot(111)
# TODO: obe razmerji v odvisnosti od p (log os x)
plt.show()
for p, c, l in zip(p_v, C_p, L_p):    # Preglednica 1
    print(f"p = {p:.1e}: C/C0 = {c:.3f}, L/L0 = {l:.3f}")

# Naloga 2 - Slika 2

In [ ]:
st_ba = np.array([Gb.stopnja(v) for v in Gb.sosedje])
G_er = nx.gnm_random_graph(Gb.n(), Gb.m(), seed=1)      # naključni graf z enakim n in m
st_er = np.array([d for _, d in G_er.degree()])
fig = plt.figure(figsize=(7, 4.5))
ax = fig.add_subplot(111)
# TODO: P(k) za BA (logaritemski razredi) in ER v log-log merilu; naklon za BA
plt.show()
# Preglednica 2
for ime, s in (("BA", st_ba), ("ER", st_er)):
    print(f"{ime}: <k> = {s.mean():.2f}, k_max = {s.max()}, delež k >= 20: {np.mean(s >= 20):.4f}")
print(f"BA: C = {povprecna_grucavost(Gb):.4f}, ER: C = {nx.average_clustering(G_er):.4f}")

# Naloga 3 - Slika 3

In [ ]:
grafi = {"obroč": obroc(60, 4), "WS $p = 0.1$": watts_strogatz(60, 4, 0.1),
         "BA $m = 1$": barabasi_albert(60, 1), "BA $m = 2$": barabasi_albert(60, 2)}
fig = plt.figure(figsize=(14, 7))
# TODO: vsak graf v krožni (zgoraj) in FR razporeditvi (spodaj)
fig.tight_layout()
plt.show()

# Naloga 4

In [ ]:
# TODO: izvoz BA grafa v CSV in JSON, ponovni uvoz in primerjava; izvoz v GraphML (networkx); Preglednica 4